# Run 6: Improved U-Net v3 (DINOv2 + CIE Lab)

Frozen DINOv2 ViT-S/14 Semantic Guide + Semantic-Guided FG-BG Cross-Attention + Hybrid Affine Residual Head + CIE Lab L1 Loss.


In [ ]:
import os, sys, zipfile, subprocess
from pathlib import Path
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])

PROJECT = Path('/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256')
DATA = Path('/content/iharmony4_256')
DATA.mkdir(parents=True, exist_ok=True)

ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']
for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = PROJECT / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

print('Dữ liệu 256x256 sẵn sàng tại:', DATA)

Mounted at /content/drive
Đang giải nén HCOCO_256.zip...
Đang giải nén HAdobe5k_256.zip...
Đang giải nén HFlickr_256.zip...
Đang giải nén Hday2night_256.zip...
Dữ liệu 256x256 sẵn sàng tại: /content/iharmony4_256


In [ ]:
import os, json, math, random, hashlib, time, platform, zipfile
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor, adjust_hue, adjust_saturation
from torchvision.utils import save_image
from pytorch_msssim import ssim
from tqdm.auto import tqdm

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & SIÊU THAM SỐ (V3: SEMANTIC-GUIDED + LAB LOSS)
# ==============================================================================
BASE_DIR = Path('/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256')
if not BASE_DIR.exists():
    BASE_DIR = Path('/content/drive/MyDrive/ImageHarmonization')

RUN = BASE_DIR / 'runs/improved_unet_v3_dinov2_lab_30ep_bs32_seed42'
RUN.mkdir(parents=True, exist_ok=True)

DATA = Path('/content/iharmony4_256')
ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']
EXTS = {'.jpg', '.jpeg', '.png'}

CFG = dict(
    model_name='improved_unet_v3_dinov2_lab',
    seed=42,
    epochs=30,
    size=256,
    batch_size=32,
    workers=2,
    base_channels=32,
    num_heads=8,
    lr=2.5e-4,
    eta_min=1e-6,
    weight_decay=1e-4,
    lambda_bg=0.1,
    lambda_lab=0.15,
    fg_color_aug_prob=0.35,  # Tăng cường lệch màu tiền cảnh để trị ca khó kiểu HFlickr #2
    val_fraction=0.05,
)

assert torch.cuda.is_available(), 'Cần bật GPU trước khi train!'
DEVICE = torch.device('cuda')

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# Giải nén dữ liệu nếu mở runtime mới
for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = BASE_DIR / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

# ==============================================================================
# 2. LẬP CHỈ MỤC & CHIA TRAIN / VAL / TEST KHỚP 100% FINGERPRINT CŨ
# ==============================================================================
def make_index(folder):
    return {p.stem: p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith('.')}

records = {'train': [], 'val': [], 'test': []}
for subset in ARCHIVES:
    subset_root = DATA / subset
    comp = make_index(subset_root / 'composite_images')
    mask = make_index(subset_root / 'masks')
    real = make_index(subset_root / 'real_images')

    subsets = {}
    for split in ('train', 'test'):
        txt = subset_root / f'{subset}_{split}.txt'
        rows = []
        for line in txt.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip():
                continue
            stem = Path(line.strip().replace('\\', '/').split()[0]).stem
            parts = stem.split('_')
            source_id, mask_id = '_'.join(parts[:-2]), '_'.join(parts[:-1])
            rows.append(dict(
                subset=subset, stem=stem, group=f'{subset}/{source_id}', full_id=f'{subset}/{stem}',
                comp=str(comp[stem]), mask=str(mask[mask_id]), real=str(real[source_id]),
            ))
        subsets[split] = rows

    groups = sorted({r['group'] for r in subsets['train']})
    random.Random(CFG['seed']).shuffle(groups)
    val_groups = set(groups[:max(1, round(len(groups) * CFG['val_fraction']))])

    for r in subsets['train']:
        records['val' if r['group'] in val_groups else 'train'].append(r)
    records['test'].extend(subsets['test'])

split_ids = {s: [r['full_id'] for r in rows] for s, rows in records.items()}
fingerprint = hashlib.sha256(json.dumps(split_ids, sort_keys=True).encode()).hexdigest()
print(f"Train: {len(records['train'])} | Val: {len(records['val'])} | Test: {len(records['test'])}")


class HarmonyDataset(Dataset):
    def __init__(self, rows, train=False):
        self.rows, self.train = rows, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        target_size = (CFG['size'], CFG['size'])
        images = []
        for key, mode in [('comp', 'RGB'), ('mask', 'L'), ('real', 'RGB')]:
            with Image.open(r[key]) as im:
                im = im.convert(mode)
                if im.size != target_size:
                    resample = Image.Resampling.NEAREST if key == 'mask' else Image.Resampling.BILINEAR
                    im = im.resize(target_size, resample)
                images.append(pil_to_tensor(im).float() / 255.)
        c, m, y = images
        m_bin = (m >= 0.5).float()
        m = m_bin if m_bin.sum() > 0 else (m > 0).float()

        if self.train:
            if torch.rand(()) < 0.5:
                c, m, y = [x.flip(-1) for x in (c, m, y)]
            # Tăng cường độ lệch tông màu chỉ ở vùng tiền cảnh để ép mạng học Semantic Cross-Attn
            if torch.rand(()) < CFG['fg_color_aug_prob']:
                hue_shift = (torch.rand(()) - 0.5).item() * 0.25       # [-0.125, 0.125]
                sat_factor = 0.7 + torch.rand(()).item() * 0.6         # [0.7, 1.3]
                c_aug = adjust_saturation(adjust_hue(c, hue_shift), sat_factor)
                c = c_aug * m + c * (1.0 - m)

        return c, m, y, r['subset']


def loader(split, epoch=0):
    g = torch.Generator().manual_seed(CFG['seed'] + epoch)
    is_train = (split == 'train')
    return DataLoader(
        HarmonyDataset(records[split], is_train),
        batch_size=CFG['batch_size'],
        shuffle=is_train,
        num_workers=CFG['workers'],
        pin_memory=True,
        drop_last=False,
        persistent_workers=(CFG['workers'] > 0),
        generator=g,
    )


# ==============================================================================
# 3. KIẾN TRÚC IMPROVED HARMONIZATION U-NET V3
# ==============================================================================
class GatedConv2d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, padding: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch * 2, kernel_size=kernel_size, padding=padding, bias=False)
        self.norm = nn.GroupNorm(num_groups=math.gcd(8, out_ch), num_channels=out_ch)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat, gate = self.conv(x).chunk(2, dim=1)
        return self.act(self.norm(feat)) * torch.sigmoid(gate)


class GatedDoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(GatedConv2d(in_ch, out_ch), GatedConv2d(out_ch, out_ch))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class StandardDoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        num_groups = math.gcd(8, out_ch)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class BilinearUp(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.up(x)


class AttentionGate(nn.Module):
    def __init__(self, g_ch: int, x_ch: int):
        super().__init__()
        inter_ch = max(x_ch // 2, 16)
        self.W_g = nn.Conv2d(g_ch, inter_ch, kernel_size=1, bias=True)
        self.W_x = nn.Conv2d(x_ch, inter_ch, kernel_size=1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(inter_ch, 1, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        return x * self.psi(self.W_g(g) + self.W_x(x))


class FrozenDINOv2Guide(nn.Module):
    def __init__(self, out_channels: int = 512):
        super().__init__()
        self.dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', verbose=False)
        for p in self.dino.parameters():
            p.requires_grad = False
        self.dino.eval()

        self.proj = nn.Sequential(
            nn.Conv2d(384, out_channels, kernel_size=1, bias=False),
            nn.GroupNorm(8, out_channels),
        )
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def train(self, mode: bool = True):
        super().train(mode)
        self.dino.eval()
        return self

    def forward(self, rgb: torch.Tensor, target_hw: tuple[int, int]) -> torch.Tensor:
        with torch.no_grad():
            x = F.interpolate(rgb, size=(224, 224), mode="bilinear", align_corners=False)
            x = (x - self.mean) / self.std
            feats = self.dino.forward_features(x)["x_norm_patchtokens"]
            B, N, C = feats.shape
            H_p = W_p = int(math.sqrt(N))
            feat_map = feats.transpose(1, 2).view(B, C, H_p, W_p)

        if (H_p, W_p) != target_hw:
            feat_map = F.interpolate(feat_map, size=target_hw, mode="bilinear", align_corners=False)
        return self.proj(feat_map)


class SemanticGuidedFGBGCrossAttention(nn.Module):
    def __init__(self, channels: int = 512, num_heads: int = 8):
        super().__init__()
        self.pos_emb = nn.Parameter(torch.randn(1, 256, channels) * 0.02)
        self.norm_sem_q = nn.LayerNorm(channels)
        self.norm_sem_k = nn.LayerNorm(channels)
        self.norm_val = nn.LayerNorm(channels)

        self.attn = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, batch_first=True)
        self.norm_ffn = nn.LayerNorm(channels)
        self.ffn = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Linear(channels * 2, channels),
        )
        self.gamma1 = nn.Parameter(torch.zeros(1))
        self.gamma2 = nn.Parameter(torch.zeros(1))

    def forward(self, spatial_feat: torch.Tensor, sem_feat: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, C, H, W = spatial_feat.shape
        m_low = F.adaptive_avg_pool2d(mask, (H, W))
        m_flat = m_low.view(B, H * W)

        sp_tokens = spatial_feat.flatten(2).transpose(1, 2)
        sem_tokens = sem_feat.flatten(2).transpose(1, 2)

        pos = self.pos_emb if H * W == self.pos_emb.shape[1] else F.interpolate(
            self.pos_emb.transpose(1, 2).view(1, C, 16, 16), size=(H, W), mode="bilinear", align_corners=False
        ).flatten(2).transpose(1, 2)

        q = self.norm_sem_q(sem_tokens + pos)
        k = self.norm_sem_k(sem_tokens + pos)
        v = self.norm_val(sp_tokens)

        bg_key_padding_mask = m_flat > 0.5
        all_fg = bg_key_padding_mask.all(dim=1)
        if all_fg.any():
            bg_key_padding_mask = bg_key_padding_mask.clone()
            bg_key_padding_mask[all_fg] = False

        attn_out, _ = self.attn(query=q, key=k, value=v, key_padding_mask=bg_key_padding_mask, need_weights=False)

        fg_gate = (m_flat > 0.2).unsqueeze(-1).to(spatial_feat.dtype)
        sp_tokens = sp_tokens + self.gamma1 * (attn_out * fg_gate)
        sp_tokens = sp_tokens + self.gamma2 * (self.ffn(self.norm_ffn(sp_tokens)) * fg_gate)
        return sp_tokens.transpose(1, 2).view(B, C, H, W)


class HybridColorAffineResidualHead(nn.Module):
    def __init__(self, in_ch: int):
        super().__init__()
        self.local_conv = nn.Conv2d(in_ch, 12, kernel_size=3, padding=1)
        self.global_fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, 12))

        self.chroma_res = nn.Sequential(
            nn.Conv2d(in_ch + 3, in_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, in_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(in_ch, 3, kernel_size=3, padding=1),
            nn.Tanh(),
        )
        self.res_scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, feat: torch.Tensor, comp: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, _, H, W = feat.shape
        params = self.local_conv(feat) + self.global_fc(feat).view(B, 12, 1, 1)
        dW = params[:, :9].view(B, 3, 3, H, W)
        db = params[:, 9:].view(B, 3, H, W)
        eye = torch.eye(3, device=feat.device, dtype=feat.dtype).view(1, 3, 3, 1, 1)

        affine_pred = torch.einsum("bijhw,bjhw->bihw", (eye + dW).float(), comp.float()) + db.float()
        delta_rgb = self.chroma_res(torch.cat([feat, affine_pred.clamp(0.0, 1.0).to(feat.dtype)], dim=1)).float()

        pred = affine_pred + self.res_scale.float() * delta_rgb * mask.float()
        return torch.clamp(pred, 0.0, 1.0).to(feat.dtype)


class ImprovedHarmonizationUNetV3(nn.Module):
    def __init__(self, base_channels: int = 32, num_heads: int = 8):
        super().__init__()
        c1, c2, c3, c4, c5 = base_channels, base_channels*2, base_channels*4, base_channels*8, base_channels*16

        self.enc1 = GatedDoubleConv(4, c1)
        self.enc2 = GatedDoubleConv(c1, c2)
        self.enc3 = StandardDoubleConv(c2, c3)
        self.enc4 = StandardDoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = StandardDoubleConv(c4, c5)
        self.sem_guide = FrozenDINOv2Guide(out_channels=c5)
        self.fg_bg_attn = SemanticGuidedFGBGCrossAttention(channels=c5, num_heads=num_heads)

        self.up4 = BilinearUp(c5, c4)
        self.att4 = AttentionGate(g_ch=c4, x_ch=c4)
        self.dec4 = StandardDoubleConv(c4 * 2, c4)

        self.up3 = BilinearUp(c4, c3)
        self.att3 = AttentionGate(g_ch=c3, x_ch=c3)
        self.dec3 = StandardDoubleConv(c3 * 2, c3)

        self.up2 = BilinearUp(c3, c2)
        self.att2 = AttentionGate(g_ch=c2, x_ch=c2)
        self.dec2 = StandardDoubleConv(c2 * 2, c2)

        self.up1 = BilinearUp(c2, c1)
        self.att1 = AttentionGate(g_ch=c1, x_ch=c1)
        self.dec1 = StandardDoubleConv(c1 * 2, c1)

        self.head = HybridColorAffineResidualHead(c1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        comp, mask = x[:, :3], x[:, 3:4]
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))

        bot_spatial = self.bottleneck(self.pool(s4))
        bot_sem = self.sem_guide(comp, target_hw=bot_spatial.shape[-2:])
        bot = self.fg_bg_attn(bot_spatial, bot_sem, mask)

        g4 = self.up4(bot)
        d4 = self.dec4(torch.cat([g4, self.att4(g4, s4)], dim=1))
        g3 = self.up3(d4)
        d3 = self.dec3(torch.cat([g3, self.att3(g3, s3)], dim=1))
        g2 = self.up2(d3)
        d2 = self.dec2(torch.cat([g2, self.att2(g2, s2)], dim=1))
        g1 = self.up1(d2)
        d1 = self.dec1(torch.cat([g1, self.att1(g1, s1)], dim=1))
        return self.head(d1, comp, mask)


# ==============================================================================
# 4. HÀM LOSS TỔNG HỢP (MASKED L1 + BG L1 + CIELAB CHROMA LOSS)
# ==============================================================================
def rgb_to_lab_ab(rgb: torch.Tensor) -> torch.Tensor:
    """Chuyển RGB [0, 1] sang 2 kênh sắc độ (a*, b*) chuẩn hóa của CIELAB (khả vi hoàn toàn)."""
    rgb = rgb.float().clamp(0.0, 1.0)
    # Inverse sRGB companding
    mask_lin = rgb > 0.04045
    rgb_lin = torch.where(mask_lin, torch.pow((rgb + 0.055) / 1.055, 2.4), rgb / 12.92)

    r, g, b = rgb_lin[:, 0:1], rgb_lin[:, 1:2], rgb_lin[:, 2:3]
    # RGB -> XYZ (D65)
    x = (0.4124564 * r + 0.3575761 * g + 0.1804375 * b) / 0.95047
    y = (0.2126729 * r + 0.7151522 * g + 0.0721750 * b) / 1.00000
    z = (0.0193339 * r + 0.1191920 * g + 0.9503041 * b) / 1.08883

    eps = 216.0 / 24389.0
    kappa = 24389.0 / 27.0
    fx = torch.where(x > eps, torch.pow(x.clamp_min(1e-8), 1.0 / 3.0), (kappa * x + 16.0) / 116.0)
    fy = torch.where(y > eps, torch.pow(y.clamp_min(1e-8), 1.0 / 3.0), (kappa * y + 16.0) / 116.0)
    fz = torch.where(z > eps, torch.pow(z.clamp_min(1e-8), 1.0 / 3.0), (kappa * z + 16.0) / 116.0)

    a = 500.0 * (fx - fy) / 128.0  # Chuẩn hóa về khoảng ~[-1, 1]
    b_ch = 200.0 * (fy - fz) / 128.0
    return torch.cat([a, b_ch], dim=1)


def compute_v3_loss(raw: torch.Tensor, target: torch.Tensor, mask: torch.Tensor, eps: float = 1e-6):
    # 1. Masked L1 (tiền cảnh)
    fg_diff = torch.abs(raw - target) * mask
    fg_l1 = (fg_diff.sum(dim=(1, 2, 3)) / (3.0 * mask.sum(dim=(1, 2, 3)).clamp_min(eps))).mean()

    # 2. Background L1 (giữ nền ổn định)
    bg_mask = 1.0 - mask
    bg_diff = torch.abs(raw - target) * bg_mask
    bg_l1 = (bg_diff.sum(dim=(1, 2, 3)) / (3.0 * bg_mask.sum(dim=(1, 2, 3)).clamp_min(eps))).mean()

    # 3. CIELAB Chroma Loss (ép đúng trục màu a*, b* trên tiền cảnh)
    pred_ab = rgb_to_lab_ab(raw)
    tgt_ab = rgb_to_lab_ab(target)
    ab_diff = torch.abs(pred_ab - tgt_ab) * mask
    lab_l1 = (ab_diff.sum(dim=(1, 2, 3)) / (2.0 * mask.sum(dim=(1, 2, 3)).clamp_min(eps))).mean()

    total = fg_l1 + CFG['lambda_bg'] * bg_l1 + CFG['lambda_lab'] * lab_l1
    return total, fg_l1, bg_l1, lab_l1


# ==============================================================================
# 5. HÀM ĐÁNH GIÁ & LƯU CHECKPOINT
# ==============================================================================
@torch.inference_mode()
def evaluate(loader_, save_to=None, include_ssim=False):
    model.eval()
    sums = defaultdict(Counter)
    for c, m, y, subsets in tqdm(loader_, desc='evaluate', leave=False):
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], 1))
        raw = raw.float()
        out = raw * m + c * (1.0 - m)
        err = (out - y).square()
        mse = err.mean((1, 2, 3))
        fmse = (err * m).sum((1, 2, 3)) / (3.0 * m.sum((1, 2, 3)).clamp_min(1.0))

        values = dict(
            raw_L1=(raw - y).abs().mean((1, 2, 3)) * 255.0,
            MSE=mse * 255**2,
            fMSE=fmse * 255**2,
            PSNR=-10 * torch.log10(mse.clamp_min(1e-12)),
            fPSNR=-10 * torch.log10(fmse.clamp_min(1e-12)),
        )
        if include_ssim:
            values['SSIM'] = ssim(out, y, data_range=1, size_average=False)

        for i, subset in enumerate(subsets):
            for label in ('all', subset):
                sums[label]['n'] += 1
                for k, v in values.items():
                    sums[label][k] += v[i].item()

        if save_to is not None:
            panels = torch.stack([c, m.expand_as(c), raw, out, y], 1).flatten(0, 1)
            save_image(panels[:30], str(save_to), nrow=5)
            save_to = None

    return {s: {k: (v if k == 'n' else v / d['n']) for k, v in d.items()} for s, d in sums.items()}


def atomic_save(obj, target):
    tmp = target.with_name(target.name + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, target)


# ==============================================================================
# 6. KHỞI TẠO & VÒNG LẶP HUẤN LUYỆN
# ==============================================================================
model = ImprovedHarmonizationUNetV3(
    base_channels=CFG['base_channels'],
    num_heads=CFG['num_heads'],
).to(DEVICE)

trainable_params = [p for p in model.parameters() if p.requires_grad]
total_m = sum(p.numel() for p in model.parameters()) / 1e6
train_m = sum(p.numel() for p in trainable_params) / 1e6
print(f"Tổng tham số: {total_m:.2f}M | Tham số huấn luyện (không tính DINOv2): {train_m:.2f}M")

optimizer = torch.optim.AdamW(trainable_params, lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=CFG['eta_min'])
scaler = torch.amp.GradScaler('cuda')

start_ep = 1
best_fmse = float('inf')
history = []

if (RUN / 'last.pt').exists():
    ck = torch.load(RUN / 'last.pt', map_location='cpu', weights_only=False)
    assert ck['cfg'] == CFG and ck['fingerprint'] == fingerprint, 'Config/data changed; use new RUN'
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)
    scheduler.load_state_dict(ck['scheduler'])
    scaler.load_state_dict(ck['scaler'])
    start_ep = ck['epoch'] + 1
    best_fmse = ck['best']
    history = ck['history']
    torch.set_rng_state(ck['torch_rng'])
    torch.cuda.set_rng_state_all(ck['cuda_rng'])
    print(f"Resume V3 từ Epoch {start_ep}/{CFG['epochs']} (best val fMSE={best_fmse:.2f})")

(RUN / 'config.json').write_text(json.dumps(dict(
    cfg=CFG,
    fingerprint=fingerprint,
    trainable_params_M=round(train_m, 2),
    total_params_M=round(total_m, 2),
    gpu=torch.cuda.get_device_name(),
    torch=torch.__version__,
    python=platform.python_version(),
), indent=2))

for ep in range(start_ep, CFG['epochs'] + 1):
    torch.cuda.synchronize()
    tic = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()

    model.train()
    tot_loss, tot_fl1, tot_bgl1, tot_lab, count = 0.0, 0.0, 0.0, 0.0, 0
    bar = tqdm(loader('train', ep), desc=f"V3 (DINO+Lab) | Ep {ep}/{CFG['epochs']}")

    for c, m, y, _ in bar:
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], dim=1))
            loss, fl1, bgl1, labl1 = compute_v3_loss(raw, y, m)

        if not torch.isfinite(loss):
            raise RuntimeError('Non-finite loss; inspect data/AMP')

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scaler.step(optimizer)
        scaler.update()

        bs = c.shape[0]
        tot_loss += loss.item() * bs
        tot_fl1 += fl1.item() * bs
        tot_bgl1 += bgl1.item() * bs
        tot_lab += labl1.item() * bs
        count += bs

        bar.set_postfix(
            loss=f'{tot_loss/count:.5f}',
            fL1=f'{tot_fl1/count:.5f}',
            lab=f'{tot_lab/count:.5f}',
        )

    assert count == len(records['train']), f'Not all training samples were seen ({count} vs {len(records["train"])})'

    scores = evaluate(loader('val'), save_to=RUN / f'val_{ep:03d}.png')
    cur_fmse = scores['all']['fMSE']
    improved = cur_fmse < best_fmse
    best_fmse = min(best_fmse, cur_fmse)

    lr_used = optimizer.param_groups[0]['lr']
    scheduler.step()
    torch.cuda.synchronize()

    row = dict(
        model='v3_dinov2_lab',
        epoch=ep,
        train_loss=tot_loss / count,
        train_fL1=tot_fl1 / count,
        train_bgL1=tot_bgl1 / count,
        train_labL1=tot_lab / count,
        val=scores,
        lr=lr_used,
        seconds=time.perf_counter() - tic,
        peak_VRAM_GB=torch.cuda.max_memory_allocated() / 2**30,
    )
    history.append(row)

    if improved:
        atomic_save(dict(model=model.state_dict(), cfg=CFG, epoch=ep, fingerprint=fingerprint, val=scores), RUN / 'best.pt')

    atomic_save(dict(
        epoch=ep,
        model=model.state_dict(),
        optimizer=optimizer.state_dict(),
        scheduler=scheduler.state_dict(),
        scaler=scaler.state_dict(),
        cfg=CFG,
        fingerprint=fingerprint,
        best=best_fmse,
        history=history,
        torch_rng=torch.get_rng_state(),
        cuda_rng=torch.cuda.get_rng_state_all(),
    ), RUN / 'last.pt')
    (RUN / 'history.json').write_text(json.dumps(history, indent=2))
    print(json.dumps(row, ensure_ascii=False))

print(f'Done V3! Best val fMSE: {best_fmse:.2f}')

Train: 62463 | Val: 3279 | Test: 7404
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 228MB/s]


Tổng tham số: 33.29M | Tham số huấn luyện (không tính DINOv2): 11.23M
Resume V3 từ Epoch 8/30 (best val fMSE=464.31)


{"model": "v3_dinov2_lab", "epoch": 8, "train_loss": 0.0689918082226228, "train_fL1": 0.06119713112119085, "train_bgL1": 0.015622520976979496, "train_labL1": 0.041549498144912766, "val": {"all": {"n": 3279, "raw_L1": 4.005331958129152, "MSE": 49.62608497694121, "fMSE": 459.7913583590876, "PSNR": 35.54410997154001, "fPSNR": 23.8255676901824}, "HCOCO": {"n": 1924, "raw_L1": 3.4589107177857303, "MSE": 26.602901170010874, "fMSE": 421.89618271750373, "PSNR": 36.76352438659033, "fPSNR": 23.48409750704458}, "HAdobe5k": {"n": 974, "raw_L1": 4.085604009197478, "MSE": 51.359562075756166, "fMSE": 385.56016646077745, "PSNR": 35.5122312490945, "fPSNR": 25.76361175043627}, "HFlickr": {"n": 368, "raw_L1": 6.466065504628679, "MSE": 155.70306098040032, "fMSE": 820.2637145000955, "PSNR": 29.55127637800963, "fPSNR": 20.719253711078476}, "Hday2night": {"n": 13, "raw_L1": 9.20375838646522, "MSE": 324.38545274734497, "fMSE": 1425.766110933744, "PSNR": 27.102517934945915, "fPSNR": 17.091510919424202}}, "lr":

... (18 outputs left out)

{"model": "v3_dinov2_lab", "epoch": 27, "train_loss": 0.05595598025111923, "train_fL1": 0.04958275977659366, "train_bgL1": 0.006153013913835119, "train_labL1": 0.038386125241531815, "val": {"all": {"n": 3279, "raw_L1": 2.7394161492122606, "MSE": 36.79012371051162, "fMSE": 362.65221038668864, "PSNR": 36.35513877868652, "fPSNR": 24.786615453430993}, "HCOCO": {"n": 1924, "raw_L1": 2.207546645601177, "MSE": 21.13669692522027, "fMSE": 332.7148682996785, "PSNR": 37.61916409510337, "fPSNR": 24.574075989316754}, "HAdobe5k": {"n": 974, "raw_L1": 2.8956702366869065, "MSE": 34.9417603843266, "fMSE": 297.7760733848235, "PSNR": 36.23447768937881, "fPSNR": 26.524610707402474}, "HFlickr": {"n": 368, "raw_L1": 5.03697996813318, "MSE": 121.46152763457403, "fMSE": 673.9915965640026, "PSNR": 30.28024002261784, "fPSNR": 21.457233947256338}, "Hday2night": {"n": 13, "raw_L1": 4.710336024944599, "MSE": 95.13030606049757, "fMSE": 840.7990951538086, "PSNR": 30.28605681199294, "fPSNR": 20.273611288804275}}, "lr

{"model": "v3_dinov2_lab", "epoch": 28, "train_loss": 0.055809008047306014, "train_fL1": 0.04944123144714967, "train_bgL1": 0.006046267353364028, "train_labL1": 0.03842099732945621, "val": {"all": {"n": 3279, "raw_L1": 2.7181068918406144, "MSE": 37.06868904128275, "fMSE": 361.543432579058, "PSNR": 36.3862732540874, "fPSNR": 24.82953504310508}, "HCOCO": {"n": 1924, "raw_L1": 2.1912954130874107, "MSE": 20.9267616004929, "fMSE": 329.9675279119654, "PSNR": 37.64347839355469, "fPSNR": 24.601084219442832}, "HAdobe5k": {"n": 974, "raw_L1": 2.852595060696592, "MSE": 35.12685279223831, "fMSE": 294.5324965918578, "PSNR": 36.29462861133552, "fPSNR": 26.619490473666964}, "HFlickr": {"n": 368, "raw_L1": 5.038154811962791, "MSE": 124.12876331093518, "fMSE": 685.26349401474, "PSNR": 30.28825108383013, "fPSNR": 21.46479086253954}, "Hday2night": {"n": 13, "raw_L1": 4.934735664954553, "MSE": 107.09250222719632, "fMSE": 891.6749420166016, "PSNR": 29.80714651254507, "fPSNR": 19.779431489797737}}, "lr": 7.

{"model": "v3_dinov2_lab", "epoch": 29, "train_loss": 0.055751096696399174, "train_fL1": 0.049396203172383316, "train_bgL1": 0.0059829755161645405, "train_labL1": 0.038377305322814706, "val": {"all": {"n": 3279, "raw_L1": 2.654759064448538, "MSE": 36.94578923459059, "fMSE": 362.26548981470245, "PSNR": 36.36075153158874, "fPSNR": 24.79276036416368}, "HCOCO": {"n": 1924, "raw_L1": 2.12847835832351, "MSE": 20.876896643651, "fMSE": 330.0258411012915, "PSNR": 37.64540898428142, "fPSNR": 24.602960794234722}, "HAdobe5k": {"n": 974, "raw_L1": 2.8025689826364144, "MSE": 35.549637951125106, "fMSE": 300.2023511109166, "PSNR": 36.197010316398355, "fPSNR": 26.484187591247245}, "HFlickr": {"n": 368, "raw_L1": 4.947482726820137, "MSE": 122.50044514886712, "fMSE": 677.6852523347605, "PSNR": 30.295762347138446, "fPSNR": 21.471722413664278}, "Hday2night": {"n": 13, "raw_L1": 4.568213719588059, "MSE": 97.89096758915828, "fMSE": 854.8892294076772, "PSNR": 30.185291877159706, "fPSNR": 20.16708491398738}}, 

{"model": "v3_dinov2_lab", "epoch": 30, "train_loss": 0.055572870851060205, "train_fL1": 0.04923160953542855, "train_bgL1": 0.005930495715694409, "train_labL1": 0.03832141004280078, "val": {"all": {"n": 3279, "raw_L1": 2.647898609644031, "MSE": 36.88699094788008, "fMSE": 361.32603488545766, "PSNR": 36.38195137775464, "fPSNR": 24.8242558551729}, "HCOCO": {"n": 1924, "raw_L1": 2.122104805187475, "MSE": 20.885319445522313, "fMSE": 329.29121097556765, "PSNR": 37.65265209808667, "fPSNR": 24.61542797782565}, "HAdobe5k": {"n": 974, "raw_L1": 2.7952894042771943, "MSE": 35.37854555803035, "fMSE": 298.15095769844993, "PSNR": 36.24663516089657, "fPSNR": 26.556886123680727}, "HFlickr": {"n": 368, "raw_L1": 4.941777372489804, "MSE": 122.49310353592686, "fMSE": 678.4454412253007, "PSNR": 30.31296169239542, "fPSNR": 21.492375049902044}, "Hday2night": {"n": 13, "raw_L1": 4.487995615372291, "MSE": 94.84024847470798, "fMSE": 858.8325617863582, "PSNR": 30.255644724919247, "fPSNR": 20.23464745741624}}, "l